In [30]:
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',  
)

model='llama3.1:8b'

In [ ]:
import re

EXPECTED_FIELDS = {
    "product_id",
    "description",
    "pros",
    "cons",
    "tags",
}

MIN_DESCRIPTION_LENGTH = 120 
MAX_DESCRIPTION_LENGTH = 700 


class ResponseValidationError(ValueError):
    """Ошибка парсинга или проверки ответа модели."""


def count_sentences(text):
    """Число предложений в тексте. Точка внутри числа концом предложения не считается."""
    return len([s for s in re.split(r"[.!?]+(?!\d)", text) if s.strip()]) 


def parse_and_validate(content: str, product_id_req: str) -> dict[str, str]:
    """Преобразовать JSON-ответ и проверить его контракт."""

    ret = {"product_id": product_id_req}

    try:
        data = json.loads(content)
    except json.JSONDecodeError as error:
        return ret | {'error': f"Ответ не является корректным JSON: {error.msg}"}

    if not isinstance(data, dict):
        return ret | {'error': "На верхнем уровне должен находиться JSON-объект"}

    received_fields = set(data)
    missing_fields = EXPECTED_FIELDS - received_fields
    unexpected_fields = received_fields - EXPECTED_FIELDS

    if missing_fields:
        return ret | {'error': "Отсутствуют обязательные поля: "
            f"{sorted(missing_fields)}"
        }

    if unexpected_fields:
        return ret | {'error': "Отсутствуют обязательные поля: "
            "Получены лишние поля: "
            f"{sorted(unexpected_fields)}"
        }

    product_id = data['product_id']
    if product_id != product_id_req:
        return ret | {'error': "Поле product_id не соответствует входным параметрам"}

    description = data["description"]

    if not isinstance(description, str):
        return ret | {'error': "Поле description должно быть строкой"}

    description = description.strip()

    if not description:
        return ret | {'error': "Поле description не должно быть пустым"}
      
    i = count_sentences(description)
    if i < 2 or i > 5:
        return ret | {'error': "Description must contain 2-5 sentences"}

    if len(description) < MIN_DESCRIPTION_LENGTH:
        return ret | {'error': "Отсутствуют обязательные поля: "
            "Поле description содержит больше "
            f"{MAX_DESCRIPTION_LENGTH} символов"
        }

    if len(description) > MAX_DESCRIPTION_LENGTH:
        return ret | {'error': "Поле description содержит меньше "
            f"{MAX_DESCRIPTION_LENGTH} символов"
        }

    pros = data['pros']
    if len(pros) < 2 or len(pros) > 5:
       return ret | {'error': "Поле pros must contain 2-5 values"}

    cons = data['cons']
    if len(cons) < 1 or len(cons) > 3:
       return ret | {'error': "Поле cons must contain 1-3 values"}
    
    tags = data['tags']
    if len(tags) < 3 or len(tags) > 8:
       return ret | {'error': "Поле tags must contain 3-8 values"}
    
    return ret

In [32]:
from typing import List
from typing_extensions import Annotated

from pydantic import (
    AfterValidator,
    BaseModel,
    ConfigDict,
    Field,
    field_validator,
)

class CardRequest(BaseModel):
    """Product Card"""

    model_config = ConfigDict(extra="forbid")

    product_id: str = Field(min_length=1, max_length=50)
    description: Annotated[str, Field(min_length=120, max_length=700), AfterValidator(count_sentences)]
    pros: Annotated[List[str], Field(min_length=2, max_length=5)]
    cons: Annotated[List[str], Field(min_length=1, max_length=3)]
    tags: Annotated[List[str], Field(min_length=3, max_length=8)]


    @field_validator("description")
    @classmethod
    def validate_description(cls, value: str) -> str:
      """Удалить внешние пробелы и проверить описание."""
      value = value.strip()

      if not value:
        raise ValueError(
            "Поле summary не должно быть пустым"
        )
      
      i = count_sentences(value)
      if i < 2 or i > 5:
        raise ValueError("Description must contain 2-5 sentences")
      
      return value


    @field_validator('pros')
    @classmethod
    def check_pros(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы pros не могут быть пустыми строками")
      return v


    @field_validator('cons')
    @classmethod
    def check_cons(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы cons не могут быть пустыми строками")
      return v


    @field_validator('tags')
    @classmethod
    def check_tags(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы tags не могут быть пустыми строками")
      return v

In [33]:
response_schema = CardRequest.model_json_schema()
response_schema

{'additionalProperties': False,
 'description': 'Product Card',
 'properties': {'product_id': {'maxLength': 50,
   'minLength': 1,
   'title': 'Product Id',
   'type': 'string'},
  'description': {'maxLength': 700,
   'minLength': 120,
   'title': 'Description',
   'type': 'string'},
  'pros': {'items': {'type': 'string'},
   'maxItems': 5,
   'minItems': 2,
   'title': 'Pros',
   'type': 'array'},
  'cons': {'items': {'type': 'string'},
   'maxItems': 3,
   'minItems': 1,
   'title': 'Cons',
   'type': 'array'},
  'tags': {'items': {'type': 'string'},
   'maxItems': 8,
   'minItems': 3,
   'title': 'Tags',
   'type': 'array'}},
 'required': ['product_id', 'description', 'pros', 'cons', 'tags'],
 'title': 'CardRequest',
 'type': 'object'}

In [47]:
def request_one(
    msg: dict, 
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350}
) -> str:
  messages=[
    {"role": "system", "content": 
      """
        Generate a product card in Russian (utf-8) by example.
        Example:
        {
          "product_id": "P-1001",
          "description": "Компактные внутриканальные наушники с Bluetooth 5.3 и защитой IPX4. Держат заряд до 24 часов и надёжно сидят при активном движении. Подойдут для спорта и повседневного использования.",
          "pros": ["Чистый звук", "Удобная посадка", "Долгое время работы"],
          "cons": ["Быстро разряжаются на морозе"],
          "tags": ["наушники", "беспроводные", "спорт", "IPX4", "Bluetooth 5.3"]
        } 
        "description": 2-5 sentences
      """
    },
    msg,
  ]
  # print('>>> messages', messages)

  response = client.chat.completions.create(
      model=model,  
      messages=messages,

      ####### seed ################## 
      seed=42, 
      
      temperature=params['temperature'],
      top_p=params['top_p'],
      max_tokens=params['max_tokens'],
      response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "support_request",
            "strict": True,
            "schema": CardRequest.model_json_schema(),
        },
      },
      extra_body={
        "provider": {
          "require_parameters": True,
          "top_k": params['top_k'], 
          "repetition_penalty": params['repetition_penalty'],
        },
      },
  )

  content = response.choices[0].message.content
  if not content:
    raise ResponseValidationError(
        "Empty response of model"
    )
  
  return content

In [84]:
# test request_one
item = {
  "product_id": "P-0027", 
  "title": "Пылесос Klin Eco", 
  "category": "Бытовая техника - Пылесосы", "attributes": {"бренд": "Klin", "тип": "с контейнером", "мощность всасывания": "220 Вт", "объём контейнера": "2 л", "время работы": "60 мин", "вес": "1.5 кг"}, 
  "reviews": ["Что могу сказать: Щётка забивается волосами, чистить неудобно", 
                "Насадки удобные, под мебель достаёт легко", 
  ]
}
row_dict = {"role": "user", "content": f"""{item}"""}

resp = request_one(row_dict)
print('>>> resp\n', resp)

>>> resp
 {
  "product_id": "P-0027",
  "description": "Пылесос Klin Eco - компактный и мощный пылесос с контейнером, идеальный для бытового использования. Обладает мощностью всасывания 220 Вт и может работать до 60 минут без перерыва. Весит всего 1,5 кг, что делает его легким в управлении.",
  "pros": ["Удобные насадки", "Легкий в управлении", "Долгое время работы"],
  "cons": ["Щётка забивается волосами", "Чистить неудобно"],
  "tags": ["пылесос", "клин", "бытовая техника", "пылесос с контейнером", "мощность всасывания 220 Вт"]
}


In [ ]:
MAX_ATTEMPTS = 3 

def generate_one(
    msg: dict, 
    product_id: str,
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350},
) -> str:
  messages_history = msg.copy()

  counter = 0
  errors = 0
  for attempt in range(1, MAX_ATTEMPTS + 1):
    counter += 1
    content = request_one(messages_history, params)

    error_dict = parse_and_validate(content, product_id)
    if not 'error' in error_dict:
      return content
    else:
      errors += 1
      print('>>> error: ', attempt, error_dict)

      if attempt == MAX_ATTEMPTS:
        return {"product_id": product_id, "error": "ResponseValidationError"}

      messages_history = [ messages_history.copy(),
        {
          "role": "assistant",
          "content": content,
        },
        {
          "role": "user",
          "content": (
              "Предыдущий ответ не прошёл проверку. "
              f"Причина: {error_dict}. "
              "Исправь ответ и верни только JSON"
          ),
        },
      ]

In [85]:
# test generate_one
item = {
  "product_id": "P-0027", 
  "title": "Пылесос Klin Eco", 
  "category": "Бытовая техника - Пылесосы", "attributes": {"бренд": "Klin", "тип": "с контейнером", "мощность всасывания": "220 Вт", "объём контейнера": "2 л", "время работы": "60 мин", "вес": "1.5 кг"}, 
  "reviews": ["Что могу сказать: Щётка забивается волосами, чистить неудобно", 
                "Насадки удобные, под мебель достаёт легко", 
  ]
}
row_dict = {"role": "user", "content": f"""{item}"""}

resp = generate_one(row_dict, product_id=item['product_id'])
print('>>> resp\n', resp)

>>>>>
 P-0027 {
  "product_id": "P-0027",
  "description": "Пылесос Klin Eco - компактный и удобный пылесос с мощностью всасывания 220 Вт. Он оснащен контейнером объёмом 2 л, что позволяет хранить большое количество пыли. Пылесос работает до 60 минут без подзарядки и весит всего 1,5 кг.",
  "pros": ["Удобные насадки", "Подходит для чистки под мебелью"],
  "cons": ["Щётка забивается волосами", "Чистить неудобно"],
  "tags": ["пылесос", "клин", "бытовая техника", "пылесос с контейнером"]
}
>>> resp
 {
  "product_id": "P-0027",
  "description": "Пылесос Klin Eco - компактный и удобный пылесос с мощностью всасывания 220 Вт. Он оснащен контейнером объёмом 2 л, что позволяет хранить большое количество пыли. Пылесос работает до 60 минут без подзарядки и весит всего 1,5 кг.",
  "pros": ["Удобные насадки", "Подходит для чистки под мебелью"],
  "cons": ["Щётка забивается волосами", "Чистить неудобно"],
  "tags": ["пылесос", "клин", "бытовая техника", "пылесос с контейнером"]
}


In [43]:
file_name = './dev.jsonl'
with open(file_name, 'r', encoding='utf-8') as file:
  items = file.read().splitlines()

In [53]:
import json
from datetime import datetime
from pathlib import Path

def wright_files(timestamp_str_dir, results, params, df):
  timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
  directory = Path(f"./outputs/{timestamp_str_dir}/{timestamp_str}")
  directory.mkdir(parents=True, exist_ok=True)

  with open(directory / "results.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4, ensure_ascii=False)

  with open(directory / "params.json", "w", encoding="utf-8") as file:
    json.dump(params, file, indent=4, ensure_ascii=False)

  totals_path = f"./outputs/{timestamp_str_dir}" / "totals.csv"
  df.to_csv(totals_path, index=False, encoding="utf-8")

In [90]:
from transformers import AutoTokenizer

def token_count(text: str, model_name: str) -> int:
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  return len(tokenizer(text)["input_ids"])

In [ ]:
import pandas as pd
import time
import json

items_slice = items[:3]

df = pd.DataFrame(columns=["temperature", "top_p", "top_k", "repetition_penalty", "max_tokens", 'error', 'descr_len', 'tokens', 'time_per_item'])
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

counter = 0
MODEL = 'meta-llama/Llama-3.1-8B-Instruct'

#   for temperature in [0.0, 0.3, 0.7]:
#     for top_p in [0.9, 1.0]:
#       for top_k in [20, 50, None]:
#         for repetition_penalty in [1.0, 1.1, 1.2]:
#           for max_tokens in [250, 350, 500]:
for temperature in [0.0]:
  for top_p in [0.9]:
    for top_k in [20]:
      for repetition_penalty in [1.0]:
        for max_tokens in [250]:
          params = {
            "temperature": temperature, 
            "top_p": top_p, 
            "top_k": top_k, 
            "repetition_penalty": repetition_penalty, 
            "max_tokens": max_tokens,
          }
          for item in items_slice:
            err = 0
            counter += 1
            row_dict = {"role": "user", "content": item}
            # print('>>> item', i)

            item_dict = json.loads(item)

            started = time.perf_counter()
            results = generate_one(row_dict, product_id=item_dict['product_id'], params=params)
            generation_time = time.perf_counter() - started

            if 'error' in results:
              err = 1
              descr_len = 0
            else:
              results_dict = json.loads(results)
              descr_len = len(results_dict['description'])
              token_count = token_count(results, MODEL)
              
            new_row = params | {'error': err, 'descr_len': descr_len, 'tokens': token_count, 'time_per_item': round(generation_time/counter, 2)}
            print(new_row)

            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            # wright_files(timestamp_str, results, new_row, df)

>>>>>
 P-0027 {
  "product_id": "P-0027",
  "description": "Пылесос Klin Eco - компактный и мощный пылесос с контейнером, идеальный для бытового использования. Обладает мощностью всасывания 220 Вт и может работать до 60 минут без перерыва. Весит всего 1,5 кг, что делает его легким и удобным в использовании.",
  "pros": ["Удобные насадки", "Работает тихо", "Легкий вес", "Большой объём контейнера"],
  "cons": ["Контейнер вытряхивается быстро", "Щётка забивается волосами"],
  "tags": ["пылесос", "клин", "бесплатная доставка", "бытовая техника", "пылесос с контейнером"]
}


OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'llama3.1:8b'.

In [87]:
df_sorted = df.sort_values(by='time_per_item')
df_sorted

,temperature,top_p,top_k,repetition_penalty,max_tokens,error,descr_len,time_per_item
2,0.0,0.9,20,1.0,250,0,233,22.77
1,0.0,0.9,20,1.0,250,0,272,54.34
0,0.0,0.9,20,1.0,250,0,248,94.88


In [88]:
totals_path = Path(f"./outputs/totals_final_{timestamp_str}.csv")
df_sorted.to_csv(totals_path, index=False, encoding="utf-8")

зафиксируйте, как каждый параметр влияет на долю валидных карточек, среднюю длину описания и число выходных токенов.